--Part 1--

In [23]:
# valve open at time=t will release flowrate*(30-t)
# destination => open the valve  # no need to go back as destination 
# current_loc => destination : distance = d; t-=d; acc+=flowrate*(30-t) 
# go through all possible paths: n!
# measure distance from any valve to any other valve => floyd warshall
# Valve["AA"].adj => {"DD":1, "II":1,"BB":1}


import re

def parse_string(string):
    #split ether "Valve " or " has flow rate=" or "; tunnels lead to valves " or "tunnel leads to valve"
    match = re.search(r'tunnels lead to valves', string)
    if match:
        string = re.split("Valve | has flow rate=|; tunnels lead to valves ", string)
    else:
        string = re.split("Valve | has flow rate=|; tunnel leads to valve ", string)
    string = [i for i in string if i]
    valve=string[0]
    flow_rate=int(string[1])
    adj=string[2].split(", ")
    return valve, flow_rate, adj 

# print(parse_string("Valve AA has flow rate=0; tunnels lead to valves DD, II, BB"))


class Valve:
    def __init__(self, name, flow_rate, adj:dict):
        self.name = name
        self.flow_rate = flow_rate
        self.adj = adj

with open('/Users/yang/Desktop/adventofcode/d16_valve.txt') as file:
    lines=file.read().splitlines()

valves = {}
for line in lines:
    valve, flow_rate, adj = parse_string(line)
    #make adj a dict with value as 1
    adj = {i:1 for i in adj}
    valves[valve] = Valve(valve, flow_rate, adj)

# fill in the distance between any two valves for floyd warshall
for i in valves.keys():
    for j in valves.keys():
        if i not in valves[j].adj:
            valves[j].adj[i] = 999

# print(valves["BB"].adj)
# print(valves["BB"].adj["HH"])

#use floyd warshall to find shortest path between any two valves
for k in valves.keys():
    for i in valves.keys():
        for j in valves.keys():
            valves[j].adj[i]=min(valves[j].adj[i], valves[j].adj[k]+valves[k].adj[i])
            valves[i].adj[j]=min(valves[j].adj[i], valves[j].adj[k]+valves[k].adj[i])

# print(valves["AA"].adj)
# print(valves["AA"].adj["HH"])



acc_list = []
path_list = []
# find all possible paths and calculate the total flow rate
# time is the time left
def find_all_paths(graph, start, time, path=[], acc=0):
    path.append(start)
    acc+=valves[start].flow_rate*(time)
    if len(path)==len(graph):
        path_list.append(path)
        acc_list.append(acc)
    else:
        for k in graph.keys():
            if k not in path:
                distance = valves[start].adj[k]
                if time-distance>=0:
                    path_copy = path.copy() #need to copy the path
                    time-=distance
                    find_all_paths(graph, k, time, path_copy, acc)
                else:
                    continue
            else:
                continue

find_all_paths(valves, "AA", 30)

print(max(acc_list))
print(path_list[acc_list.index(max(acc_list))])

            
            


1893
['AA', 'BB', 'CC', 'DD', 'EE', 'FF', 'GG', 'HH', 'II', 'JJ']


In [1]:
#Valve AA has flow rate=0; tunnels lead to valves DD, II, BB
#dict{AA:[0,status,[DD,II,BB]]}
#create a dictionary of valves and their flow rates and tunnels

#or
#class valve: 
#self.flow_rate
#self.status: 0 for closed , 1 for open
#self.nexttunnels = [DD,II,BB]
"""
import re


#make a function to parse the string
def parse_string(string):
    #split ether "Valve " or " has flow rate=" or "; tunnels lead to valves " or "tunnel leads to valve"
    match = re.search(r'tunnels lead to valves', string)
    if match:
        string = re.split("Valve | has flow rate=|; tunnels lead to valves ", string)
    else:
        string = re.split("Valve | has flow rate=|; tunnel leads to valve ", string)
    string = [i for i in string if i]
    valve=string[0]
    flow_rate=int(string[1])
    adj=string[2].split(", ")
    return valve, flow_rate, adj 

print(parse_string("Valve AA has flow rate=0; tunnels lead to valves DD, II, BB"))


class Valve:
    def __init__(self,flow_rate, adj:list, status=0):
        self.flow_rate = flow_rate
        self.status = status
        self.adj = adj

    # def update_status(self,valve_dict, time):
    #     if self.status == 0:


# def update_flow(current_flow,added_flow): 
#     current_flow+=added_flow        
#     return current_flow



with open('/Users/yang/Desktop/adventofcode/d16_valve.txt') as file:
    lines=file.read().splitlines()

# myfile=open("/Users/yang/Desktop/adventofcode/d16_valve.txt",'r')
# lines=myfile.readlines()
#parse string
#make a dictionary of valves and their flow rates and tunnels
valve_dict={}

for i in range(len(lines)):
    parsed=parse_string(lines[i])
    print(parsed)
    valve_dict[parsed[0]]=Valve(parsed[1],parsed[2])

# for i in valve_dict:
#     print("\n",i,"flow rate:",valve_dict[i].flow_rate)
#     print("status:",valve_dict[i].status)
#     print("adj:",valve_dict[i].adj)
"""

('AA', 0, ['DD', 'II', 'BB'])
('AA', 0, ['DD', 'II', 'BB'])
('BB', 13, ['CC', 'AA'])
('CC', 2, ['DD', 'BB'])
('DD', 20, ['CC', 'AA', 'EE'])
('EE', 3, ['FF', 'DD'])
('FF', 0, ['EE', 'GG'])
('GG', 0, ['FF', 'HH'])
('HH', 22, ['GG'])
('II', 0, ['AA', 'JJ'])
('JJ', 21, ['II'])


In [ ]:
#floyd warshall algorithm


In [1]:
testlist=["AA","BB","CC"]
testlist["AA"]

TypeError: list indices must be integers or slices, not str

In [2]:
#brute force
#not working because dictionary is passed by reference
"""
current_pos="AA"
time=30
acc_list=[]
current_flow=0

#dictionary is passed by reference!!!!!!!!!!!!! each branch of recursion should have its own copy of dictionary
#can stop recursion when all valves are open

def walk_through_tunnels(valve_dict, current_pos, time, current_flow, acc):
    time-=1
    acc+=current_flow
    
    if time>0:
        next_tunnels=valve_dict[current_pos].adj
        #check if status is open
        if valve_dict[current_pos].status==1:
            for i in next_tunnels:
                current_pos = i
                walk_through_tunnels(valve_dict, current_pos, time, current_flow, acc) 
        else:
            #to open the valve or not
            for k in range(2):
                if k==0:
                    valve_dict[current_pos].status=1
                    current_flow+=valve_dict[current_pos].flow_rate  #dont know why this line is not working
                    print("update flow:",current_flow)
                    # for key in valve_dict:
                    #     print(key, valve_dict[key].status)
                    walk_through_tunnels(valve_dict, current_pos, time, current_flow, acc)
                else:
                    for i in next_tunnels:
                        current_pos = i
                        walk_through_tunnels(valve_dict, current_pos, time, current_flow, acc)
    else:
        acc_list.append(acc)
        #print everyhing 100 times
        if len(acc_list)%100==0:
            print(len(acc_list))
            print(acc_list[1],acc_list[-1])
            print("current_flow:",current_flow)
            for key in valve_dict:
                print(key, valve_dict[key].status)
    



walk_through_tunnels(valve_dict, current_pos, 5, 0, 0)
acc_list=sorted(acc_list)
print(len(acc_list))
print(acc_list[1],acc_list[-1])
"""


update flow: 0
update flow: 20
update flow: 22
update flow: 23
update flow: 0
update flow: 21
update flow: 13
update flow: 0
65
0 43


In [ ]:

# while time>0:
#     next_tunnels=valve_dict[current_pos].adj
#     #check if status is open
#     if valve_dict[current_pos].status==1:
#         for i in next_tunnels:
#             current_pos = i
#             time-=1
#             #recursion
#     else:
#         #to open the valve or not
#         for k in range(2):
#             if k==0:
#                 valve_dict[current_pos].status=1
#                 acc+=update_status(valve_dict)
#                 time-=1
#             else:
#                 for i in next_tunnels:
#                     current_pos = i
#                     time-=1
#                     #recursion

# make a recursive function to calculate the acc after 30 seconds

    


In [15]:
s=("AA","BB","CC")
s=s+("DD",)
print(s)

('AA', 'BB', 'CC', 'DD')


In [52]:
regex1 = r"Valve\s"
regex2 = r"\shas flow rate="
regex3 = r";\stunnel\w* lead\w* to valve\w*\s"
print("test:",re.split(regex3, "Valve AA has flow rate=0; tunnels lead to valves DD, II, BB"))
print("test:",re.split(regex3, "Valve HH has flow rate=22; tunnel leads to valve GG"))

test: ['Valve AA has flow rate=0', 'DD, II, BB']
test: ['Valve HH has flow rate=22', 'GG']
